# 10 Pseudobulk

Create donor-level pseudobulk count tables for broad classes, glia subtypes, and selected excitatory subtypes.

The default path is memory-conscious: reuse the existing `adata_merged_pb_26donors.h5ad` checkpoint and stream sparse count rows from disk. Set `REBUILD_SOURCE_OBJECT = True` only when the annotation/full-gene source objects have changed.


## 1. Imports, paths and settings


In [1]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import scanpy as sc

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

RESULTS_DIR = PROJECT_ROOT / "results"
PSEUDOBULK_DIR = RESULTS_DIR / "10_Pseudobulk_26donors"
TABLES_DIR = PSEUDOBULK_DIR / "tables"

ADATA_PB_PATH = PROJECT_ROOT / "data" / "processed" / "merged" / "adata_merged_pb_26donors.h5ad"
HM_ANNOT_PATH = PROJECT_ROOT / "data" / "processed" / "merged" / "adata_hm_annotated_26donors.h5ad"
FULL_ANNOT_PATH = PROJECT_ROOT / "data" / "processed" / "merged" / "adata_full_annotated_26donors.h5ad"

OUT_DIR = {
    "broad": PSEUDOBULK_DIR / "broad",
    "glia_subtypes": PSEUDOBULK_DIR / "glia_subtypes",
    "excitatory_subtypes": PSEUDOBULK_DIR / "excitatory_subtypes",
}

for path in [PSEUDOBULK_DIR, TABLES_DIR, *OUT_DIR.values()]:
    path.mkdir(parents=True, exist_ok=True)

In [9]:
REBUILD_SOURCE_OBJECT = False
COUNT_LAYER = "counts"
MIN_CELLS_PER_DONOR_GROUP = 5

BROAD_GROUPS = {
    "Excitatory": ["Excitatory"],
    "Inhibitory": ["Inhibitory"],
    "Glia": ["Glia"],
    "Vascular": ["Vascular"],
}

GLIA_GROUPS = {
    "Glia_Oligo": ["Glia.Oligo"],
    "Glia_OPC": ["Glia.OPC"],
    "Glia_Micro": ["Glia.Micro"],
    "Glia_Astro": ["Glia.Astro.GFAP.neg", "Glia.Astro.GFAP.pos"],
}

EXCITATORY_GROUPBY_COL = "cell_subtype_with_UMN_like"

EXCITATORY_GROUPS = {
    "Ex_L5_PCP4_NXPH2": ["Ex.L5.PCP4_NXPH2"],
    "Ex_L5_PCP4_NXPH2_UMN_like": ["Ex.L5.PCP4_NXPH2.UMN_like"],
    "Ex_L5_L6_THEMIS_NR4A2": ["Ex.L5_L6.THEMIS_NR4A2"],
    "Ex_L5_L6_THEMIS_TMEM233": ["Ex.L5_L6.THEMIS_TMEM233"],
    "Ex_L6_TLE4_MEGF11": ["Ex.L6.TLE4_MEGF11"],
    "Ex_L6_TLE4_CCBE1": ["Ex.L6.TLE4_CCBE1"],
}

## 2. Optional: rebuild the pseudobulk source object


In [10]:
if REBUILD_SOURCE_OBJECT:
    adata_merged_pb = sc.read_h5ad(FULL_ANNOT_PATH)

    required_cols = [
        "leiden_harmony",
        "cell_class_major_harmony",
        "cell_subtype",
        "cell_subtype_with_UMN_like",
        "UMN_like_high_confidence",
        "sample",
        "condition",
    ]
    missing_cols = [col for col in required_cols if col not in adata_merged_pb.obs.columns]
    if missing_cols:
        raise KeyError(f"Missing required annotation columns in {FULL_ANNOT_PATH}: {missing_cols}")

    if "soupx_counts" in adata_merged_pb.layers:
        adata_merged_pb.layers[COUNT_LAYER] = adata_merged_pb.layers["soupx_counts"].copy()
    elif "raw_counts" in adata_merged_pb.layers:
        adata_merged_pb.layers[COUNT_LAYER] = adata_merged_pb.layers["raw_counts"].copy()
    else:
        adata_merged_pb.layers[COUNT_LAYER] = adata_merged_pb.X.copy()

    adata_merged_pb.write_h5ad(ADATA_PB_PATH)

    print("Saved:", ADATA_PB_PATH)
    print(adata_merged_pb)
    print(adata_merged_pb.obs["condition"].value_counts())
    print(adata_merged_pb.obs["cell_class_major_harmony"].value_counts(dropna=False))
    print(adata_merged_pb.obs["cell_subtype"].value_counts(dropna=False))
    print(adata_merged_pb.obs["cell_subtype_with_UMN_like"].value_counts(dropna=False))
    print("High-confidence UMN-like cells:", int(adata_merged_pb.obs["UMN_like_high_confidence"].sum()))
else:
    print("Using existing checkpoint:", ADATA_PB_PATH)



Saved: /nemo/lab/schreibera/home/users/wangj/FlowCytometry/c9_multiomics/data/processed/merged/adata_merged_pb_26donors.h5ad
AnnData object with n_obs × n_vars = 163784 × 61552
    obs: 'sample', 'condition', 'dataset', 'tissue', 'source_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outliers', 'mt_outliers', 'high_quality_cells', 'soupx_clusters', 'doublet_score_scrublet', 'predicted_doublet_scrublet', 'barcode', 'merge_key', 'leiden_harmony', 'cell_class_major_harmony', 'cell_subtype', 'UMN_strict_detected', 'UMN_like_high_confidence', 'cell_subtype_with_UMN_like'
    var: 'gene_ids', 'feature_types', 'mt', 'ribo', 'hb'
    obsm: 'X_pca_harmony', 'X_umap'
    layers: 'counts', 'raw_counts', 'soupx_counts'
condition
sALS   

## 3. Stream pseudobulk counts from the checkpoint


In [6]:
def read_string_array(obj):
    values = obj[:]
    return np.array([v.decode("utf-8") if isinstance(v, bytes) else str(v) for v in values])


def read_obs_column(obs_group, column):
    obj = obs_group[column]
    if hasattr(obj, "keys") and "codes" in obj and "categories" in obj:
        return read_string_array(obj["categories"])[obj["codes"][:]]
    return read_string_array(obj)


def sum_csr_rows(csr_group, row_indices, n_genes):
    data = csr_group["data"]
    indices = csr_group["indices"]
    indptr = csr_group["indptr"]
    summed = np.zeros(n_genes, dtype=np.float64)
    for row in row_indices:
        start, end = indptr[row], indptr[row + 1]
        np.add.at(summed, indices[start:end], data[start:end])
    return summed


def save_group_panel(h5, obs, var_names, groups, out_dir, groupby_col, output_group_col="group"):
    counts_layer = h5[f"layers/{COUNT_LAYER}"]
    saved, skipped, retention_rows = [], [], []

    for group_name, labels in groups.items():
        mask = obs[groupby_col].isin(labels)
        obs_group = obs.loc[mask, ["sample", "condition", groupby_col]].copy()

        if obs_group.empty:
            skipped.append({"group_name": group_name, "reason": "no_cells_found"})
            continue

        count_rows, meta_rows = [], []
        for (sample, condition), sample_obs in obs_group.groupby(["sample", "condition"], sort=True):
            row_indices = sample_obs.index.to_numpy()
            n_cells = len(row_indices)
            if n_cells < MIN_CELLS_PER_DONOR_GROUP:
                continue
            count_rows.append(pd.Series(sum_csr_rows(counts_layer, row_indices, len(var_names)), index=var_names, name=sample))
            meta_rows.append({
                "sample": sample,
                "condition": condition,
                "group": group_name,
                "source_labels": ",".join(labels),
                "groupby_col": output_group_col,
                "n_cells": n_cells,
            })

        if not count_rows:
            skipped.append({"group_name": group_name, "reason": "no_donor_groups_passed_min_cells"})
            continue

        pd.DataFrame(count_rows).to_csv(out_dir / f"{group_name}_counts.csv")
        meta = pd.DataFrame(meta_rows).set_index("sample")
        meta.to_csv(out_dir / f"{group_name}_meta.csv")
        saved.append(group_name)

        tmp = meta.reset_index().copy()
        tmp["group_name"] = group_name
        retention_rows.append(tmp[["group_name", "sample", "condition", "n_cells"]])

    retention = pd.concat(retention_rows, ignore_index=True) if retention_rows else pd.DataFrame(columns=["group_name", "sample", "condition", "n_cells"])
    skipped = pd.DataFrame(skipped) if skipped else pd.DataFrame(columns=["group_name", "reason"])
    return saved, retention, skipped


In [11]:
with h5py.File(ADATA_PB_PATH, "r") as h5:
    var_names = read_string_array(h5["var"]["_index"])
    obs = pd.DataFrame({
        "sample": read_obs_column(h5["obs"], "sample"),
        "condition": read_obs_column(h5["obs"], "condition"),
        "cell_class_major_harmony": read_obs_column(h5["obs"], "cell_class_major_harmony"),
        "cell_subtype": read_obs_column(h5["obs"], "cell_subtype"),
        "cell_subtype_with_UMN_like": read_obs_column(h5["obs"], "cell_subtype_with_UMN_like"),
    })

    print("Broad labels:")
    print(obs["cell_class_major_harmony"].value_counts(dropna=False))

    print("\nSubtype labels:")
    print(obs["cell_subtype"].value_counts(dropna=False))

    broad_saved, broad_retention, broad_skipped = save_group_panel(
        h5, obs, var_names, BROAD_GROUPS, OUT_DIR["broad"], "cell_class_major_harmony", "cell_class_major_harmony"
    )
    glia_saved, glia_retention, glia_skipped = save_group_panel(
        h5, obs, var_names, GLIA_GROUPS, OUT_DIR["glia_subtypes"], "cell_subtype", "cell_subtype"
    )
    exc_saved, exc_retention, exc_skipped = save_group_panel(
        h5,
        obs,
        var_names,
        EXCITATORY_GROUPS,
        OUT_DIR["excitatory_subtypes"],
        EXCITATORY_GROUPBY_COL,
        EXCITATORY_GROUPBY_COL,
    )

    print("\nSubtype labels with UMN-like category:")
    print(obs["cell_subtype_with_UMN_like"].value_counts(dropna=False))

Broad labels:
cell_class_major_harmony
Glia          117152
Excitatory     32043
Inhibitory     13197
Vascular        1392
Name: count, dtype: int64

Subtype labels:
cell_subtype
Glia.Oligo                 87096
Ex.L2_L3.CUX2_RASGRF2      14924
Glia.Astro.GFAP.neg        14239
Glia.OPC                    9120
Glia.Micro                  6697
Ex.L4_L5.RORB_POU3F2        4945
In.5HT3aR.DISC1_RELN        4798
Ex.L4_L6.RORB_LRRK1         3671
Ex.L5_L6.THEMIS_TMEM233     3660
In.PV.PVALB_MYBPC1          3379
In.SOM.SST_ADAMTS19         3191
In.Rosehip.LAMP5_PMEPA1     1829
Ex.L6.TLE4_MEGF11           1651
Ex.L6.TLE4_CCBE1            1480
Ex.L5.PCP4_NXPH2             872
Ex.L5_L6.THEMIS_NR4A2        840
Vasc.Endo.Venous             782
Vasc.Mural.Pericyte          610
Name: count, dtype: int64

Subtype labels with UMN-like category:
cell_subtype_with_UMN_like
Glia.Oligo                   87096
Ex.L2_L3.CUX2_RASGRF2        14924
Glia.Astro.GFAP.neg          14239
Glia.OPC                     

## 4. Retention summaries and output inventory


In [13]:
import scanpy as sc
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/nemo/lab/schreibera/home/users/wangj/FlowCytometry/c9_multiomics")
FULL_ANNOT_PATH = PROJECT_ROOT / "data" / "processed" / "merged" / "adata_full_annotated_26donors.h5ad"

adata_full = sc.read_h5ad(FULL_ANNOT_PATH)

umn = adata_full.obs[
    adata_full.obs["UMN_like_high_confidence"].astype(bool)
].copy()

print("Total high-confidence UMN-like cells:", umn.shape[0])

print("\nUMN-like cells by condition:")
display(umn["condition"].astype(str).value_counts())

print("\nUMN-like cells by sample and condition:")
umn_by_sample = (
    umn.groupby(["condition", "sample"])
    .size()
    .rename("n_umn_like_cells")
    .reset_index()
    .sort_values(["condition", "n_umn_like_cells"], ascending=[True, False])
)
display(umn_by_sample)

Total high-confidence UMN-like cells: 65

UMN-like cells by condition:


condition
Control    28
c9ALS      28
sALS        9
Name: count, dtype: int64


UMN-like cells by sample and condition:


/tmp/slurm_48994769/ipykernel_1185533/89354747.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  umn.groupby(["condition", "sample"])


,condition,sample,n_umn_like_cells
17,Control,GSM5292176,9
21,Control,GSM5292180,6
18,Control,GSM5292177,3
15,Control,GSM5292174,2
16,Control,GSM5292175,2
...,...,...,...
73,sALS,GSM5292180,0
74,sALS,GSM5292181,0
75,sALS,GSM5292182,0
76,sALS,GSM5292183,0


In [14]:
for threshold in [1, 5, 10, 20]:
    retained = umn_by_sample[umn_by_sample["n_umn_like_cells"] >= threshold]
    print(f"\nThreshold >= {threshold} cells")
    display(
        retained.groupby("condition")["sample"]
        .nunique()
        .rename("n_retained_donors")
        .reset_index()
    )


Threshold >= 1 cells


/tmp/slurm_48994769/ipykernel_1185533/2061013937.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  retained.groupby("condition")["sample"]


,condition,n_retained_donors
0,Control,8
1,c9ALS,5
2,sALS,7



Threshold >= 5 cells


/tmp/slurm_48994769/ipykernel_1185533/2061013937.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  retained.groupby("condition")["sample"]


...

/tmp/slurm_48994769/ipykernel_1185533/2061013937.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  retained.groupby("condition")["sample"]


,condition,n_retained_donors
0,Control,0
1,c9ALS,1
2,sALS,0



Threshold >= 20 cells


/tmp/slurm_48994769/ipykernel_1185533/2061013937.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  retained.groupby("condition")["sample"]


,condition,n_retained_donors
0,Control,0
1,c9ALS,0
2,sALS,0


In [12]:
retention_outputs = {
    "broad": broad_retention,
    "glia": glia_retention,
    "excitatory": exc_retention,
}
skipped_outputs = {
    "broad": broad_skipped,
    "glia": glia_skipped,
    "excitatory": exc_skipped,
}

for name, retention in retention_outputs.items():
    retention.to_csv(TABLES_DIR / f"{name}_pseudobulk_retention_by_donor.csv", index=False)
    summary = (
        retention.groupby(["group_name", "condition"])
        .agg(n_retained_donors=("sample", "nunique"), min_cells=("n_cells", "min"), median_cells=("n_cells", "median"), max_cells=("n_cells", "max"))
        .reset_index()
        if not retention.empty else pd.DataFrame()
    )
    summary.to_csv(TABLES_DIR / f"{name}_pseudobulk_retention_summary.csv", index=False)
    skipped_outputs[name].to_csv(TABLES_DIR / f"{name}_pseudobulk_skipped_groups.csv", index=False)

pd.DataFrame(
    [{"target_group": k, "source_cell_subtype": v} for k, labels in GLIA_GROUPS.items() for v in labels]
).to_csv(TABLES_DIR / "glia_subtype_mapping.csv", index=False)

pd.DataFrame(
    [
        {
            "target_group": k,
            "source_cell_subtype_with_UMN_like": v,
        }
        for k, labels in EXCITATORY_GROUPS.items()
        for v in labels
    ]
).to_csv(TABLES_DIR / "excitatory_subtype_mapping.csv", index=False)

inventory = []
for level, out_dir in OUT_DIR.items():
    for path in sorted(out_dir.glob("*.csv")):
        inventory.append({"level": level, "file": str(path.relative_to(PROJECT_ROOT))})
inventory = pd.DataFrame(inventory)
inventory.to_csv(TABLES_DIR / "pseudobulk_output_inventory.csv", index=False)
inventory


,level,file
0,broad,results/10_Pseudobulk_26donors/broad/Excitator...
1,broad,results/10_Pseudobulk_26donors/broad/Excitator...
2,broad,results/10_Pseudobulk_26donors/broad/Glia_coun...
3,broad,results/10_Pseudobulk_26donors/broad/Glia_meta...
4,broad,results/10_Pseudobulk_26donors/broad/Inhibitor...
5,broad,results/10_Pseudobulk_26donors/broad/Inhibitor...
6,broad,results/10_Pseudobulk_26donors/broad/Vascular_...
7,broad,results/10_Pseudobulk_26donors/broad/Vascular_...
8,glia_subtypes,results/10_Pseudobulk_26donors/glia_subtypes/G...
9,glia_subtypes,results/10_Pseudobulk_26donors/glia_subtypes/G...
